# Text Operations



The output-parameter pattern used below is explained in {ref}`Chapter 5 parameter passing <gap-return-values-out-and-ref>`.


```{contents}
:local:
:depth: 2
```


```{index} Treat Each Imported Line as a Small Contract
```



```{index} Treat Each Imported Line as a Small Contract
```

## Treat Each Imported Line as a Small Contract



An inventory file contains code, quantity and unit price separated by commas. Splitting a line gives pieces of text; it does not prove that there are exactly three fields, that a quantity is an integer, or that a price is nonnegative. Work from the schema toward conversion, not from a convenient Parse call toward assumptions about everything that came before it.

For a deliberately simple format without quoted separators, process a line through distinct stages. Preserve the original text and line number so that a rejected record can be explained. First count fields, then normalize only fields whose rules permit it, then parse numeric representations, and finally check the business domain. A zero quantity may be allowed even when an empty quantity field is not.

| Line | First relevant result |
| --- | --- |
| PEN,3,1.25 | Three fields; numeric and domain checks can proceed |
| PEN,three,1.25 | Quantity representation is invalid |
| PEN,-3,1.25 | Quantity is numeric but violates a nonnegative rule |
| PEN,3 | Schema is incomplete |

The order matters. Indexing the third field before checking field count can throw instead of producing a useful rejection. Normalizing every character in the entire line can alter meaningful free text. Catching every exception can hide a programming defect such as an incorrect index as though it were bad source data.

A regular expression can describe the shape of one field, such as a fixed prefix followed by digits. It does not replace business validation. A code may match a shape and still refer to an unknown product. For a small fixed delimiter format, Split and ordinary conditions may be easier to understand. The supporting regex reading develops groups, replacement and pattern-specific syntax; use a pattern only after stating what language of text it should accept.

Guided exercise: design acceptance/rejection outcomes for the four lines above, an empty line, and quantity zero. Decide whether one rejected line cancels the import or is reported while valid lines continue. Then predict the final accepted count under your chosen policy. Compare the prepared AI import candidate against that explicit policy instead of trusting a plausible explanation of its code.

For related explanation, see the further reading below.[^development-reading]


This section applies the string methods from {ref}`string-methods` (Section 5.4) to whole lines of text: splitting lines into fields, validating them, cleaning text safely, and reading structured records.

As a reminder, strings are immutable, so methods such as `Trim`, `ToUpper`, and `Replace` return new strings. Clean a line with `Trim()` and check it with `string.IsNullOrWhiteSpace()` before splitting or parsing it.


```{index} Splitting and Tokenizing
```



```{index} Splitting and Tokenizing
```

## Splitting and Tokenizing



Splitting turns one text line into meaningful fields. `Split` takes the separator and returns an array with one string per field, in order:

```csharp
string[] parts = line.Split(',');   // one element per comma-separated field
```

The three fields of `Ada,98,CS` land at indexes 0, 1, and 2. Note that `parts[1]` is the *text* `"98"`, not a number, until the program converts it:


In [ ]:
string line    = "Ada,98,CS";
string[] parts = line.Split(',');

Console.WriteLine(parts[0]);  // Ada
Console.WriteLine(parts[1]);  // 98
Console.WriteLine(parts[2]);  // CS

The output is `Ada`, `98`, and `CS`. Each field is a separate string, and the commas themselves are gone. A line with a different number of commas produces a different array length, which is why the validation step below checks `parts.Length` before indexing.


Input often has extra whitespace — trim each field after splitting:

In [ ]:
string line = " Ada , 98 , CS ";
string[] parts = line.Split(',');

for (int i = 0; i < parts.Length; i++)
    parts[i] = parts[i].Trim();

foreach (var p in parts)
    Console.WriteLine($"'{p}'");

The output is `'Ada'`, `'98'`, and `'CS'`, with the quotes showing that the surrounding spaces are gone. `Split` keeps whitespace inside each field, so `" 98 "` would fail `int.TryParse` until it is trimmed; trimming every field once, right after the split, is cheaper than remembering to do it at every use.


When input may contain repeated delimiters, remove empty entries:

In [ ]:
string line = "Ada,,CS";
string[] parts = line.Split(',', StringSplitOptions.RemoveEmptyEntries);

Console.WriteLine(parts.Length);  // 2

The output is `2`. The option discarded the empty middle field, which is convenient for lines padded with repeated commas but dangerous for records with a fixed field order: `parts[1]` is now `CS`, not the missing score, so a later `parts[2]` throws and `parts[1]` silently names the wrong column. For fixed-format records, keep the empty entries and treat them as missing values instead.


```{index} Token Validation
```



```{index} Token Validation
```

### Token Validation




Validate the number of fields before accessing by index:

In [ ]:
string line = "Ada,98,CS";
string[] parts = line.Split(',');

if (parts.Length != 3)
{
    Console.WriteLine("Invalid record: expected 3 fields.");
}
else
{
    Console.WriteLine($"Name: {parts[0].Trim()}, Score: {parts[1].Trim()}, Major: {parts[2].Trim()}");
}

---

```{index} Safe Replacement Strategy
```



```{index} Safe Replacement Strategy
```

## Safe Replacement Strategy



Replacing text is safe when it cannot destroy information you still need:

1. Preserve the original text. Strings are immutable, so `Replace` already returns a new string; keep the variable that refers to the original.
2. Create a cleaned copy and work on that.
3. Validate the output before trusting it, for example by parsing the replaced field.
4. Write to a new file first, then swap it into place, so a failure part way through leaves the original intact.

```csharp
string updated = original.Replace(oldText, newText);   // every occurrence; original is unchanged
```


In [ ]:
string original = "Score: N/A";
string updated  = original.Replace("N/A", "0");

Console.WriteLine($"original: {original}");
Console.WriteLine($"updated:  {updated}");

The output is `original: Score: N/A` and `updated:  Score: 0`. `Replace` is case-sensitive and replaces every occurrence, so `"N/A, N/A"` would become `"0, 0"`. The mistake to avoid is calling `original.Replace(...)` without assigning the result: the original is untouched and the cleaned text is thrown away.


---

```{index} Structured Lines
```



```{index} Structured Lines
```

## Structured Lines




Structured lines are text records with predictable field order.

Example record: `Alice,98,CS`

Field model:
1. name (`string`)
2. score (`int`)
3. major (`string`)

A parser for one record follows the `TryParse` shape from Chapter 6: it returns `true` with the fields delivered through `out` parameters, or `false` with safe defaults, and it never throws for a malformed line. Three checks guard the three ways a line can be wrong: blank, wrong field count, and a score that is not a number.


In [ ]:
// The parser on its own; the next cell repeats it so that it can run standalone.
static bool TryParseStudent(string line, out string name, out int score, out string major)
{
    name  = "";
    major = "";
    score = 0;

    if (string.IsNullOrWhiteSpace(line)) return false;

    string[] parts = line.Split(',');
    if (parts.Length != 3) return false;

    name  = parts[0].Trim();
    major = parts[2].Trim();

    return int.TryParse(parts[1].Trim(), out score);
}

In [ ]:
using System;

string[] records = { "Alice,98,CS", "Bob,abc,Math", "Carol,87,Physics", "" };

foreach (string line in records)
{
    if (TryParseStudent(line, out string name, out int score, out string major))
        Console.WriteLine($"{name} ({major}) scored {score}");
    else
        Console.WriteLine($"Skipped malformed line: '{line}'");
}

static bool TryParseStudent(string line, out string name, out int score, out string major)
{
    name  = "";
    major = "";
    score = 0;

    if (string.IsNullOrWhiteSpace(line)) return false;

    string[] parts = line.Split(',');
    if (parts.Length != 3) return false;

    name  = parts[0].Trim();
    major = parts[2].Trim();

    return int.TryParse(parts[1].Trim(), out score);
}


The standalone demonstration repeats its helper deliberately. It prints Alice and Carol as accepted records and reports Bob and the blank line as malformed. Bob fails integer parsing; the blank line has no record. This helper checks field count and score syntax, not every possible business rule: for example, it does not reject an empty name or a score outside a course's permitted range. The prepared stock-import review below adds explicit domain rules; a successful parse alone does not prove the record is valid for every application.


The output is `Alice (CS) scored 98`, `Skipped malformed line: 'Bob,abc,Math'`, `Carol (Physics) scored 87`, and `Skipped malformed line: ''`. Bob's line has three fields but `abc` fails the integer parse, so the method returns `false` after having set `name` and `major`; the caller must not use any of the outputs when the result is `false`. Variation: add a fourth field to Alice's line and predict which check rejects it.


```{index} When to Use a Database
```

## When to Use a Database




```{index} database; choosing persistence
```


Text files work well for exports and small sequential datasets. When related records need repeated queries, constraints, and coordinated updates, consider a relational database. After classes and object-oriented design, the optional {doc}`Databases and Bookstore Inventory appendix <../appendices/databases/index>` shows a complete SQLite application and connected inventory lab.

| Need | Text file | Relational database |
| --- | --- | --- |
| Read every record once, in order | Natural | Possible, more setup |
| Find one record by key, often | Scan the file each time | Indexed lookup |
| Keep two related record sets consistent | Program must enforce it | Constraints enforce it |
| Several programs updating at once | Risky | Transactions coordinate it |
| Hand the data to another tool | Universal | Needs an export, often to text |


```{index} AI Review: Specify and Validate a Text Import
```

## AI Review: Specify and Validate a Text Import




```{index} AI assistance; synthetic data; schema; validation; text import
```


This is a prepared review example, not a transcript from a named model. Predict and review it yourself first. If your course permits AI assistance, use the suggested prompt to ask for an explanation or additional checks. You can complete the entire activity with the supplied candidate and your own reasoning; no AI account is required.

A stock import uses exactly two comma-separated fields: a nonblank SKU and an integer quantity from zero through 500 inclusive. Trim the fields. This small exercise does not support quoted CSV fields or embedded commas. A missing/extra field, blank SKU, malformed number, or out-of-range quantity must be rejected.

A prepared suggestion says to split the line, read `fields[1]`, and call `int.Parse`. Explain why that can fail on a missing field or malformed quantity, and why a successfully parsed negative number still violates the contract.

**Possible AI prompt:** “Review a parser for this two-field stock schema. Use only these synthetic examples. Identify missing validation and propose test cases; do not add CSV quoting support or change the business range.” Supply a schema and a few fictional records rather than uploading a real customer or inventory export.

Complete the parser below. Set the output values only after validating all fields; initialize them to empty text and zero on rejection. Predict which three records will be accepted before running.


In [ ]:
// Exercise: Validate a prepared import design
using System;

static bool TryReadStockLine(string line, out string sku, out int quantity)
{
    sku = "";
    quantity = 0;
    // Your code starts here.
    return false; // TODO: validate field count, text, type, and business range.
    // Your code ends here.
}

// Synthetic stock data; no customer records are used.
string[] lines =
{
    "SKU-1042,12", "SKU-1043,0", "SKU-1044,500",
    "SKU-1045,-1", "SKU-1046,501", "SKU-1047,12x",
    "broken", ",3", "SKU-1048,5,extra"
};

foreach (string line in lines)
{
    if (TryReadStockLine(line, out string sku, out int quantity))
        Console.WriteLine($"ACCEPT {sku}: {quantity}");
    else
        Console.WriteLine($"REJECT [{line}]");
}


In [1]:
// Solution: Validate a prepared import design
using System;

static bool TryReadStockLine(string line, out string sku, out int quantity)
{
    sku = "";
    quantity = 0;
    // Your code starts here.
    string[] fields = line.Split(',');
    if (fields.Length != 2) return false;
    string candidateSku = fields[0].Trim();
    if (string.IsNullOrWhiteSpace(candidateSku)) return false;
    if (!int.TryParse(fields[1].Trim(), out int candidateQuantity)) return false;
    if (candidateQuantity < 0 || candidateQuantity > 500) return false;
    sku = candidateSku;
    quantity = candidateQuantity;
    return true;
    // Your code ends here.
}

// Synthetic stock data; no customer records are used.
string[] lines =
{
    "SKU-1042,12", "SKU-1043,0", "SKU-1044,500",
    "SKU-1045,-1", "SKU-1046,501", "SKU-1047,12x",
    "broken", ",3", "SKU-1048,5,extra"
};

foreach (string line in lines)
{
    if (TryReadStockLine(line, out string sku, out int quantity))
        Console.WriteLine($"ACCEPT {sku}: {quantity}");
    else
        Console.WriteLine($"REJECT [{line}]");
}


ACCEPT SKU-1042: 12
ACCEPT SKU-1043: 0
ACCEPT SKU-1044: 500
REJECT [SKU-1045,-1]
REJECT [SKU-1046,501]
REJECT [SKU-1047,12x]
REJECT [broken]
REJECT [,3]
REJECT [SKU-1048,5,extra]


The accepted quantities are 12, zero, and 500. Explain why an AI suggestion that deletes every rejected line from the original file would introduce another problem: you would lose evidence needed to inspect or repair the input. Preserve the original, validate a copy, and inspect the report before writing changes. In this exercise the rejection report echoes synthetic stock lines; avoid copying that choice into logs containing personal data.

Record the schema supplied for review, one rejected suggestion or missing check, and the input/output evidence supporting your parser. If you used a live assistant, note the tool and date and summarize which advice you accepted or rejected. Do not include credentials or private records in that account.


```{rubric} Footnotes
```


[^development-reading]: Further reading: [Nakov and colleagues, Strings and Text Processing](https://introprogramming.info/english-intro-csharp-book/read-online/chapter-13-strings-and-text-processing/). The worked business scenarios and guided reasoning here are original teaching examples.
